# Modern Databases 2026, practice: a cinema chain

This notebook works through one scenario in the order of the course: an EER diagram, a relational schema, CREATE TABLE statements with constraints, loading the data, and SQL queries. There is no model solution. For every query, the notebook shows the result that a correct query returns, so that you can check your own.

Do Parts 1 and 2 on paper, as in the exam, before you look at the data in Part 4. The notebook needs DuckDB 1.5.5 and six CSV files of about 0.8 MB in total, which the setup downloads from the course repository, [github.com/JanKalo/MDB2026](https://github.com/JanKalo/MDB2026), unless they are already next to the notebook. The cinemas, films and people are fictional.

| Part | Task |
|---|---|
| 1 | Draw an EER diagram for the scenario. |
| 2 | Translate it into a relational schema. |
| 3 | Write the CREATE TABLE statements. |
| 4 | Load the chain's data into your tables. |
| 5 | Test your constraints with 15 changes. |
| 6 | Write 17 queries and repair one. |

## The scenario

A small cinema chain runs three cinemas, in Amsterdam, Utrecht and Haarlem. So far it has kept its programme and its ticket sales in spreadsheets. It now wants a database.

Every cinema has a unique name and a unique three-letter code, such as NRD for Cinema Noord, and it is located in a city. A cinema has several screens, numbered 1, 2, 3 and so on within the cinema. Within a screen, a seat is identified by its row letter and its seat number, so seat F12 exists in many screens. Every seat is a standard seat, a love seat (a double seat that is sold as one ticket) or a wheelchair space. The website shows the capacity of every screen, which is its number of seats.

A film is identified by its title together with its release year, because some titles have been used for more than one film. For every film, the chain stores the running time in minutes, the minimum age according to the Dutch Kijkwijzer (all ages, 6, 9, 12, 14, 16 or 18) and one or more genres. A film can be the sequel of one earlier film.

A screening shows one film in one screen, starting at a given time. A screen never has two screenings that start at the same time, and two screenings in the same screen must not overlap. Every screening has a base price for a ticket.

Customers register with their email address, which identifies them, and with their name and date of birth. A registered customer can become a member. For members, the chain stores since when they are a member and their home cinema.

A ticket is for one seat at one screening, and a seat can be sold only once for the same screening. For every ticket, the chain stores the price paid and the moment of sale, which lies before the start of the screening. A ticket bought online belongs to the registered customer who bought it. A ticket bought at the box office belongs to nobody. Customers may only buy tickets for films whose minimum age they have reached on the day of the screening.

Registered customers can rate a film they have seen, once, with one to five stars. The date of the rating is stored.

## Part 1: EER diagram

Draw an EER diagram for the scenario, on paper or in draw.io, in the notation of the lectures: (min,max) at both ends of every relationship, with * for many, keys underlined and partial keys with a dashed underline. Use the identifiers the chain already has; the scenario needs no invented id numbers.

Then write down the rules from the scenario that your diagram cannot express.

There is no model diagram. Parts 3 to 5 test your design: the data has to fit into your tables, and your tables have to accept and reject the changes in Part 5 as listed.

## Part 2: relational schema

Translate your diagram into relations in the notation of the lectures, for example R(<u>a</u>, (b, c) → S, d): underline the primary key and write a composite foreign key as one reference in brackets.

Which rules from Part 1 does your relational schema still not enforce? Which of them could a CHECK constraint enforce once the tables exist?

## Setup

Run the next three cells once. The first installs DuckDB 1.5.5. The second finds the six data files next to the notebook or downloads them into a folder `cinema-data`. The third opens an empty database and defines five helpers:

- `run(...)` runs statements such as CREATE TABLE and INSERT.
- `q(...)` runs one query and shows its result as a table.
- `counts()` lists your tables with their number of rows.
- `attempt(...)` tries a change for Part 5 and undoes it again.
- `reset()` drops everything you created, so that you can start Part 3 again.

The database is kept in memory, so after a restart you run the notebook from the top and your own cells rebuild your tables.

In [ ]:
%pip -q install duckdb==1.5.5 pandas

In [ ]:
import os
import urllib.request

import duckdb
import pandas as pd

print("DuckDB", duckdb.__version__)
assert duckdb.__version__.startswith("1.5"), "Install DuckDB 1.5.5: pip install duckdb==1.5.5 (do not upgrade during the course)"
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 160)

FILES = ["seats.csv", "films.csv", "programme.csv", "customers.csv", "tickets.csv", "ratings.csv"]
BASE = "https://raw.githubusercontent.com/JanKalo/MDB2026/main/data/cinema/"   # the course repository
DOWNLOADS = "cinema-data"

def locate(name):
    for folder in [".", "cinema", "data/cinema", "../data/cinema", DOWNLOADS]:
        candidate = os.path.join(folder, name)
        if os.path.exists(candidate):
            return candidate.replace("\\", "/")
    os.makedirs(DOWNLOADS, exist_ok=True)
    target = os.path.join(DOWNLOADS, name)
    try:
        urllib.request.urlretrieve(BASE + name, target)
    except OSError as error:
        if os.path.exists(target):
            os.remove(target)
        raise RuntimeError(f"Could not download {BASE + name} ({error}). Download the six CSV files "
                           "from data/cinema in the course repository and put them next to this notebook.") from None
    return target.replace("\\", "/")

paths = {name: locate(name) for name in FILES}
paths

In [ ]:
con = duckdb.connect()   # an empty database in memory

def _empty(sql):
    return not any(line.split("--")[0].strip() for line in sql.splitlines())

def run(sql):
    """Run statements such as CREATE TABLE and INSERT."""
    if _empty(sql):
        print("This cell has no statement yet.")
        return
    con.execute(sql)
    print("Done.")

def q(sql):
    """Run one query and show its result as a table."""
    if _empty(sql):
        print("This cell has no query yet.")
        return None
    result = con.sql(sql)
    if result is None:
        print("Done. This statement returns no table.")
        return None
    return result.df()

def counts():
    """List your tables with their number of rows."""
    names = [row[0] for row in con.execute(
        "SELECT table_name FROM duckdb_tables() WHERE NOT temporary ORDER BY table_name").fetchall()]
    rows = [(name, con.execute(f'SELECT COUNT(*) FROM "{name}"').fetchone()[0]) for name in names]
    return pd.DataFrame(rows, columns=["table", "rows"])

def attempt(sql):
    """Try a change, say whether the database accepted it, then undo it."""
    if _empty(sql):
        print("This cell has no statement yet.")
        return
    con.execute("BEGIN TRANSACTION")
    try:
        con.execute(sql)
        print("accepted")
    except duckdb.Error as error:
        print("rejected:", str(error).splitlines()[0])
    finally:
        con.execute("ROLLBACK")

def reset():
    """Drop everything you created (tables, views, sequences, types), so that you can run Part 3 again."""
    for _ in range(30):
        views = [r[0] for r in con.execute(
            "SELECT view_name FROM duckdb_views() WHERE NOT internal AND NOT temporary").fetchall()]
        tables = [r[0] for r in con.execute(
            "SELECT table_name FROM duckdb_tables() WHERE NOT temporary").fetchall()]
        if not views and not tables:
            break
        for name in views:
            con.execute(f'DROP VIEW IF EXISTS "{name}"')
        for name in tables:
            try:
                con.execute(f'DROP TABLE "{name}"')
            except duckdb.Error:
                pass   # another table still refers to it; it goes in a later round
    for (name,) in con.execute("SELECT sequence_name FROM duckdb_sequences() WHERE NOT temporary").fetchall():
        con.execute(f'DROP SEQUENCE IF EXISTS "{name}"')
    for (name,) in con.execute("SELECT type_name FROM duckdb_types() WHERE NOT internal").fetchall():
        con.execute(f'DROP TYPE IF EXISTS "{name}"')
    print("Everything you created is dropped.")

## Part 3: CREATE TABLE

Write one CREATE TABLE statement per relation of Part 2 into the next cell and run it. Declare every rule of the scenario that CREATE TABLE can express: PRIMARY KEY, FOREIGN KEY, NOT NULL, UNIQUE and CHECK. Use VARCHAR for text, INTEGER for whole numbers, DATE for days, TIMESTAMP for moments and DECIMAL(5,2) for prices.

Three things about DuckDB:

- Create a table before the tables that refer to it.
- DuckDB does not support referential actions such as ON DELETE CASCADE, so leave them out.
- A CHECK can use LIKE, IN (...), BETWEEN and functions such as `regexp_full_match(column, '[0-9]+')`.

If the cell stops with an error halfway, the tables created before the error already exist, so running the cell again fails with "already exists". Run `reset()` and then the cell again. Do the same when you change a table later, and rerun your INSERT cell from Part 4 afterwards.

In [ ]:
run("""
-- Part 3: your CREATE TABLE statements

""")

In [ ]:
counts()

## Part 4: load the data

The chain exported its spreadsheets as six CSV files; the programme covers the two weeks from Thursday 3 to Wednesday 16 September 2026. The next cell makes each file available as a view, with the column types already set. Empty cells are read as NULL, and the column cinema holds the three-letter code.

| View | One row per | Columns |
|---|---|---|
| export_seats | seat | cinema, cinema_name, city, screen, seat_row, seat_no, seat_type (standard, love seat or wheelchair) |
| export_films | film | title, release_year, runtime_min, min_age (0 means all ages), genres (separated by semicolons), sequel_of_title, sequel_of_year |
| export_programme | screening | cinema, screen, start_time, title, release_year, base_price |
| export_customers | registered customer | email, name, birth_date, member_since, home_cinema (the last two only for members) |
| export_tickets | ticket | cinema, screen, start_time, seat_row, seat_no, email (empty at the box office), price_paid, sold_at |
| export_ratings | rating | email, title, release_year, stars, rated_on |

Fill your tables with INSERT INTO ... SELECT ... FROM export_... statements. Every row of the exports meets the rules that CREATE TABLE can express, so if an INSERT is rejected, the problem is in your schema or in your INSERT. The spreadsheets were never checked against the other rules, and some rows break them; questions 16 and 17 find those rows.

Two hints:

- `string_split(genres, ';')` turns the text into a list, and `unnest(string_split(genres, ';'))` in the SELECT list returns one row per genre.
- DuckDB checks a foreign key that points into the same table against the rows that were there before the statement started. If one of your tables refers to itself, first insert the rows that refer to no other row, in a separate statement.

Checkpoint: the data holds 3 cinemas, 9 screens, 747 seats, 23 films with 43 genres between them (a film with two genres counts twice), 476 screenings, 1,205 registered customers of whom 201 are members, 9,316 tickets and 802 ratings. Compare these numbers with `counts()`.

In [ ]:
EXPORTS = {
    "seats": {"cinema": "VARCHAR", "cinema_name": "VARCHAR", "city": "VARCHAR", "screen": "INTEGER", "seat_row": "VARCHAR", "seat_no": "INTEGER", "seat_type": "VARCHAR"},
    "films": {"title": "VARCHAR", "release_year": "INTEGER", "runtime_min": "INTEGER", "min_age": "INTEGER", "genres": "VARCHAR", "sequel_of_title": "VARCHAR", "sequel_of_year": "INTEGER"},
    "programme": {"cinema": "VARCHAR", "screen": "INTEGER", "start_time": "TIMESTAMP", "title": "VARCHAR", "release_year": "INTEGER", "base_price": "DECIMAL(5,2)"},
    "customers": {"email": "VARCHAR", "name": "VARCHAR", "birth_date": "DATE", "member_since": "DATE", "home_cinema": "VARCHAR"},
    "tickets": {"cinema": "VARCHAR", "screen": "INTEGER", "start_time": "TIMESTAMP", "seat_row": "VARCHAR", "seat_no": "INTEGER", "email": "VARCHAR", "price_paid": "DECIMAL(5,2)", "sold_at": "TIMESTAMP"},
    "ratings": {"email": "VARCHAR", "title": "VARCHAR", "release_year": "INTEGER", "stars": "INTEGER", "rated_on": "DATE"},
}

def make_exports():
    for name, columns in EXPORTS.items():
        spec = ", ".join(f"'{column}': '{kind}'" for column, kind in columns.items())
        con.execute(f"CREATE OR REPLACE TEMP VIEW export_{name} AS "
                    f"SELECT * FROM read_csv('{paths[name + '.csv']}', header = true, columns = {{{spec}}})")

make_exports()
q("SELECT * FROM export_films")

In [ ]:
run("""
-- Part 4: your INSERT INTO ... SELECT ... FROM export_... statements

""")

In [ ]:
counts()

## Part 5: test your constraints

Each line describes a change to the loaded data. Write it as one or more INSERT statements for your tables and run them with `attempt(...)`, which says whether the database accepted or rejected the change and then undoes it, so your data stays as loaded.

| | Change | Expected |
|---:|---|---|
| 1 | Add a cinema with the code NRD, the name Cinema Centrum and the city Amsterdam. | rejected |
| 2 | Add a cinema with the code HLM, the name Cinema Spaarne and the city Haarlem. | rejected |
| 3 | Add screen 0 to Cinema Noord (NRD). | rejected |
| 4 | Add a standard seat A1 to screen 3 of Cinema Spaarne (SPA). | rejected |
| 5 | Add seat M1 to screen 1 of Cinema Noord as a VIP seat. | rejected |
| 6 | Add the film Blue Hour (2026) with a running time of 95 minutes, minimum age 15 and the genre Drama. | rejected |
| 7 | Add the film Harbour Lights: Origins (2010) with a running time of 110 minutes, minimum age 12 and the genre Drama, as the sequel of Harbour Lights (2018). | rejected |
| 8 | Add a screening of Cloudberry (2026) in screen 2 of Cinema Spaarne on 12 September 2026 at 15:15, base price 8.50. | rejected |
| 9 | For the screening in screen 4 of Cinema Noord on 12 September 2026 at 19:45, the box office sells seat C3 for 13.50 at 19:30. | rejected |
| 10 | For the screening in screen 1 of Cinema Noord on 12 September 2026 at 19:00, the box office sells seat C5 of screen 2 for 13.50 at 18:45. | rejected, or impossible to write down |
| 11 | For the same screening in screen 1, the box office sells seat C7 for 13.50 at 19:05. | rejected |
| 12 | For the same screening in screen 1, the box office sells seat C8 for 13.50 at 18:40. | accepted |
| 13 | The customer anna.dejong@example.net rates Cloudberry (2026) with 0 stars on 20 September 2026. | rejected |
| 14 | Register Kim de Wit (kim.dewit@example.com, born 12 April 2001) as a member since 20 September 2026, without a home cinema. | rejected |
| 15 | Add a screening of The Drowned Polder (2026) in screen 1 of Cinema Noord on 12 September 2026 at 19:30, base price 13.50. | accepted |

If your tables offer no way to write down line 10, your design already rules it out, and that counts as rejected. Lines 12 and 15 must be accepted. Line 15 breaks a rule of the scenario that no CHECK constraint can express, because a CHECK tests one row at a time. Question 16 finds such cases in the data.

In [ ]:
attempt("""
-- 1

""")

In [ ]:
attempt("""
-- 2

""")

In [ ]:
attempt("""
-- 3

""")

In [ ]:
attempt("""
-- 4

""")

In [ ]:
attempt("""
-- 5

""")

In [ ]:
attempt("""
-- 6

""")

In [ ]:
attempt("""
-- 7

""")

In [ ]:
attempt("""
-- 8

""")

In [ ]:
attempt("""
-- 9

""")

In [ ]:
attempt("""
-- 10

""")

In [ ]:
attempt("""
-- 11

""")

In [ ]:
attempt("""
-- 12

""")

In [ ]:
attempt("""
-- 13

""")

In [ ]:
attempt("""
-- 14

""")

In [ ]:
attempt("""
-- 15

""")

## Part 6: queries

Answer the questions on your own tables, not on the export views. Each question shows the result that a correct query returns. Your column names may differ; the rows and values must be the same, in the order the question asks for. Questions 1 to 3 are warm-ups, 4 to 12 are at exam level, 13 to 17 are harder, and question 18 asks you to repair a query.

DuckDB syntax you may need:

- `TIMESTAMP '2026-09-12 00:00'` is a moment. For all moments of one day, compare with >= and < against two such moments.
- `start_time + INTERVAL (runtime) MINUTE` adds a number of minutes to a moment. `birth_date + INTERVAL (16) YEAR` is the day someone turns 16.
- `CAST(start_time AS DATE)` is the day of a moment.
- `ROUND(x, 2)` rounds to two decimals.

### Question 1

Which screenings are there in Haarlem on Saturday 12 September 2026? Return one row per screening: the start time, the screen number, the title and the running time, ordered by start time.

Expected result, 7 rows:

| start time | screen | title | running time |
|---|---:|---|---:|
| 2026-09-12 14:00:00 | 1 | Fog over Texel | 115 |
| 2026-09-12 15:15:00 | 2 | Two on a Bicycle | 90 |
| 2026-09-12 16:45:00 | 1 | Fog over Texel | 115 |
| 2026-09-12 18:00:00 | 2 | Seven Bridges | 112 |
| 2026-09-12 19:30:00 | 1 | Harbour Lights: Low Tide | 131 |
| 2026-09-12 20:45:00 | 2 | Salt and Iron | 165 |
| 2026-09-12 22:15:00 | 1 | Salt and Iron | 165 |

In [ ]:
q("""
-- Question 1

""")

### Question 2

How many tickets were sold at the box office?

Expected result, 1 row:

| box-office tickets |
|---:|
| 4146 |

In [ ]:
q("""
-- Question 2

""")

### Question 3

A screening ends at its start time plus the running time of its film. Which screenings that start on Friday 11 September 2026 end after midnight? Return the cinema code, the screen number, the start time, the title and the end time, ordered by end time and then by cinema code.

Expected result, 5 rows:

| cinema | screen | start time | title | end time |
|---|---:|---|---|---|
| SPA | 1 | 2026-09-11 22:15:00 | The Glass Orchard | 2026-09-12 00:23:00 |
| NRD | 4 | 2026-09-11 22:00:00 | Salt and Iron | 2026-09-12 00:45:00 |
| OGR | 1 | 2026-09-11 22:00:00 | Salt and Iron | 2026-09-12 00:45:00 |
| NRD | 2 | 2026-09-11 22:15:00 | The Cartographer Returns | 2026-09-12 00:46:00 |
| OGR | 2 | 2026-09-11 22:30:00 | Orbit of Ash | 2026-09-12 01:22:00 |

In [ ]:
q("""
-- Question 3

""")

### Question 4

Which films have at least 10 screenings? Return the title, the release year and the number of screenings, with the most screenings first and ties ordered by title.

Expected result, 14 rows:

| title | release year | screenings |
|---|---:|---:|
| Harbour Lights: Low Tide | 2026 | 55 |
| The Glass Orchard | 2026 | 51 |
| Polder Punks Forever | 2026 | 49 |
| Fog over Texel | 2025 | 44 |
| Seven Bridges | 2025 | 35 |
| The Cartographer Returns | 2025 | 32 |
| Island Crossing | 2026 | 29 |
| Salt and Iron | 2025 | 29 |
| Cloudberry | 2026 | 27 |
| Paper Kites | 2026 | 23 |
| Signal Lost | 2026 | 22 |
| Two on a Bicycle | 2025 | 19 |
| Orbit of Ash | 2026 | 15 |
| The Dike Keepers | 2026 | 10 |

In [ ]:
q("""
-- Question 4

""")

### Question 5

Which members did not buy a single ticket? Return the email address and the name, ordered by email address.

Expected result, 8 rows:

| email | name |
|---|---|
| lieke.bouzid@example.com | Lieke Bouzid |
| lieke.debruijn@example.net | Lieke de Bruijn |
| marit.dewit@example.org | Marit de Wit |
| merel.dewit@example.net | Merel de Wit |
| niels.janssen@example.com | Niels Janssen |
| priya.vermeulen@example.com | Priya Vermeulen |
| thijs.haddad@example.com | Thijs Haddad |
| yara.deboer@example.net | Yara de Boer |

In [ ]:
q("""
-- Question 5

""")

### Question 6

How many tickets were sold per genre? A ticket for a film with two genres counts for both genres. Return the genre and the number of tickets, with the most tickets first.

Expected result, 12 rows:

| genre | tickets |
|---|---:|
| Drama | 4513 |
| Thriller | 3142 |
| Crime | 2498 |
| Comedy | 1796 |
| Romance | 1645 |
| Adventure | 1562 |
| Science fiction | 1033 |
| Family | 1023 |
| Animation | 817 |
| History | 434 |
| Horror | 230 |
| Documentary | 92 |

In [ ]:
q("""
-- Question 6

""")

### Question 7

For every film in the database: how many ratings does it have, and what is its average number of stars, rounded to two decimals? A film without ratings appears with 0 ratings. Return the title, the release year, the number of ratings and the average, ordered by title and release year.

Expected result, 23 rows:

| title | release year | ratings | average stars |
|---|---:|---:|---:|
| Cloudberry | 2026 | 63 | 4.25 |
| Fog over Texel | 2025 | 57 | 3.47 |
| Harbour Lights | 2018 | 5 | 3.80 |
| Harbour Lights: Low Tide | 2026 | 128 | 4.10 |
| Island Crossing | 2026 | 40 | 3.83 |
| Moonlit Dunes | 2006 | 0 | NULL |
| Moonlit Dunes 2 | 2016 | 0 | NULL |
| Night Train to Leiden | 1979 | 2 | 4.00 |
| Night Train to Leiden | 2025 | 11 | 3.27 |
| Orbit of Ash | 2026 | 62 | 3.94 |
| Paper Kites | 2026 | 24 | 3.75 |
| Polder Punks | 2019 | 4 | 3.50 |
| Polder Punks Forever | 2026 | 85 | 3.42 |
| Salt and Iron | 2025 | 37 | 3.46 |
| Seven Bridges | 2025 | 37 | 3.54 |
| Signal Lost | 2026 | 37 | 3.73 |
| The Cartographer | 1995 | 8 | 4.25 |
| The Cartographer Returns | 2025 | 58 | 3.21 |
| The Dike Keepers | 2026 | 6 | 3.50 |
| The Drowned Polder | 2026 | 22 | 3.14 |
| The Glass Orchard | 2026 | 97 | 3.80 |
| The Quiet Canal | 2024 | 4 | 4.25 |
| Two on a Bicycle | 2025 | 15 | 3.60 |

NULL shows up as NaN or None in the notebook.

In [ ]:
q("""
-- Question 7

""")

### Question 8

For every cinema: how many tickets were sold, and what percentage of them was bought at the box office, rounded to one decimal? Order the result by cinema code.

Expected result, 3 rows:

| cinema | tickets | box office (%) |
|---|---:|---:|
| NRD | 4796 | 39.7 |
| OGR | 2933 | 47.2 |
| SPA | 1587 | 54.1 |

In [ ]:
q("""
-- Question 8

""")

### Question 9

Which screenings were sold out, with a ticket for every seat of their screen? Return the cinema code, the screen number, the start time and the title, ordered by start time.

Expected result, 3 rows:

| cinema | screen | start time | title |
|---|---:|---|---|
| SPA | 2 | 2026-09-05 20:45:00 | Harbour Lights: Low Tide |
| OGR | 3 | 2026-09-11 20:15:00 | The Glass Orchard |
| NRD | 4 | 2026-09-12 19:45:00 | Harbour Lights: Low Tide |

In [ ]:
q("""
-- Question 9

""")

### Question 10

Which sequels were released at least 10 years after the film they continue? Return the title and release year of the earlier film, the title and release year of the sequel, and the number of years in between, largest gap first.

Expected result, 2 rows:

| earlier film | year | sequel | year | years between |
|---|---:|---|---:|---:|
| The Cartographer | 1995 | The Cartographer Returns | 2025 | 30 |
| Moonlit Dunes | 2006 | Moonlit Dunes 2 | 2016 | 10 |

In [ ]:
q("""
-- Question 10

""")

### Question 11

Which customers bought tickets in every cinema of the chain? Do not write the number 3 into your query: it has to stay correct when the chain opens a fourth cinema. Return the email address and the name, ordered by email address.

Expected result, 6 rows:

| email | name |
|---|---|
| chen.vanvliet@example.net | Chen van Vliet |
| gijs.brouwer38@example.net | Gijs Brouwer |
| karim.elamrani@example.net | Karim El Amrani |
| lucas.vandermeer@example.com | Lucas van der Meer |
| tess.dehaan@example.com | Tess de Haan |
| yusuf.jacobs@example.net | Yusuf Jacobs |

In [ ]:
q("""
-- Question 11

""")

### Question 12

Which film sold the most tickets in each cinema? If several films share the maximum in a cinema, return all of them. Return the cinema code, the title, the release year and the number of tickets, ordered by cinema code and title.

Expected result, 4 rows:

| cinema | title | release year | tickets |
|---|---|---:|---:|
| NRD | Harbour Lights: Low Tide | 2026 | 955 |
| OGR | Polder Punks Forever | 2026 | 426 |
| OGR | The Glass Orchard | 2026 | 426 |
| SPA | Harbour Lights: Low Tide | 2026 | 356 |

In [ ]:
q("""
-- Question 12

""")

### Question 13

Which members bought at least one ticket, but none at their home cinema? Return the email address, the name and the code of the home cinema, ordered by email address.

Expected result, 6 rows:

| email | name | home cinema |
|---|---|---|
| emma.dijkstra@example.com | Emma Dijkstra | NRD |
| kees.hoekstra@example.org | Kees Hoekstra | NRD |
| lotte.koster@example.org | Lotte Koster | OGR |
| merel.dijkstra17@example.org | Merel Dijkstra | SPA |
| tim.deboer@example.com | Tim de Boer | OGR |
| yusuf.nowak@example.org | Yusuf Nowak | OGR |

In [ ]:
q("""
-- Question 13

""")

### Question 14

For every cinema: how many screenings did it have, and how many tickets did a screening sell on average, rounded to two decimals? A screening without any ticket counts with 0 tickets. Order the result by cinema code.

Expected result, 3 rows:

| cinema | screenings | average tickets |
|---|---:|---:|
| NRD | 224 | 21.41 |
| OGR | 154 | 19.05 |
| SPA | 98 | 16.19 |

In [ ]:
q("""
-- Question 14

""")

### Question 15

A customer has seen a film if they bought a ticket for one of its screenings. Which customers have seen a film and also its sequel? Return the email address, the title of the earlier film and the title of the sequel, once per combination, ordered by email address and then by the title of the earlier film.

Expected result, 10 rows:

| email | earlier film | sequel |
|---|---|---|
| ahmed.brouwer@example.com | Harbour Lights | Harbour Lights: Low Tide |
| chloe.huisman@example.com | The Cartographer | The Cartographer Returns |
| chloe.peters@example.org | Polder Punks | Polder Punks Forever |
| eva.debruijn@example.com | The Cartographer | The Cartographer Returns |
| maud.prins@example.com | Harbour Lights | Harbour Lights: Low Tide |
| mei.prins@example.net | Polder Punks | Polder Punks Forever |
| pien.deboer30@example.net | The Cartographer | The Cartographer Returns |
| priya.nowak@example.net | Polder Punks | Polder Punks Forever |
| tomasz.schouten@example.com | Polder Punks | Polder Punks Forever |
| yusuf.kok@example.com | Harbour Lights | Harbour Lights: Low Tide |

In [ ]:
q("""
-- Question 15

""")

### Question 16

A screening occupies its screen from its start time until its start time plus the running time of its film. The next screening may start at exactly that moment. Which pairs of screenings in the same screen overlap? Return every pair once: the cinema code, the screen number, the start time and title of the earlier screening, and the start time and title of the later one, ordered by cinema code, screen number and the earlier start time.

Expected result, 2 rows:

| cinema | screen | earlier start | earlier film | later start | later film |
|---|---:|---|---|---|---|
| NRD | 1 | 2026-09-05 19:00:00 | Orbit of Ash | 2026-09-05 21:45:00 | Island Crossing |
| OGR | 3 | 2026-09-15 17:30:00 | The Cartographer Returns | 2026-09-15 19:50:00 | Signal Lost |

In [ ]:
q("""
-- Question 16

""")

### Question 17

Which online tickets were bought by customers who had not yet reached the minimum age of the film on the day of the screening? Someone born on 5 September 2010 reaches 16 on 5 September 2026. Return the email address, the title, the start time, the date of birth and the minimum age, ordered by start time.

Expected result, 4 rows:

| email | title | start time | date of birth | minimum age |
|---|---|---|---|---:|
| ruben.bos@example.net | The Drowned Polder | 2026-09-05 22:15:00 | 2009-12-01 | 18 |
| mila.dekker@example.com | Night Train to Leiden | 2026-09-10 19:45:00 | 2011-05-05 | 16 |
| sam.visser@example.com | Signal Lost | 2026-09-10 22:15:00 | 2010-11-20 | 16 |
| lina.kaya@example.org | The Drowned Polder | 2026-09-12 22:00:00 | 2012-03-02 | 18 |

In [ ]:
q("""
-- Question 17

""")

### Question 18: repair a query

The query below should return the members who did not buy a single ticket, the same as Question 5, but it returns no rows. Explain why, then repair it. The table and column names belong to one possible schema; translate them to yours.

```sql
SELECT c.email, c.name
FROM customer AS c
WHERE c.member_since IS NOT NULL
  AND c.email NOT IN (SELECT t.email FROM ticket AS t)
ORDER BY c.email;
```

Repaired, the query returns the 8 rows of Question 5.

In [ ]:
q("""
-- Question 18, repaired

""")

## When a result differs

1. What does one row of your result represent: a film, a screening, a ticket, a customer or a group?
2. Did a join multiply rows, or did rows without a partner disappear?
3. Is every film identified by its title together with its release year?
4. Did a comparison with NULL evaluate to UNKNOWN? A ticket bought at the box office has no customer.
5. Is the condition about rows (WHERE) or about groups (HAVING)?
6. If several rows share the maximum, are all of them returned?
7. Is a boundary on the right side: a birthday on the day of the screening, a screening that starts when the previous one ends, a gap of exactly 10 years?

Go back to the logical order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY rather than editing syntax at random.